# Module 21 — Week 10 — Bayesian Black-Box Optimisation Capstone

**W9: 3/8 improved (F1 ⭐, F5 ⭐, F7 ⭐ — all new bests). F8 missed by 0.0005. Fix: F2 recovery to W6 exact, F3 x3 corrected toward W1, F6 recovery to W8, F8 ultra-tight.**

In [ ]:
import matplotlib
matplotlib.use('Agg')
import numpy as np
import matplotlib.pyplot as plt
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import Matern
from scipy.stats import norm, yeojohnson
from scipy.stats.qmc import LatinHypercube
import warnings, os
warnings.filterwarnings('ignore')
PLOTS_DIR = '/Users/luckydhanvi/Documents/DataScience/PracticalLive/LEVEL_2/imperial-aiml-capstone/module-21/plots'
os.makedirs(PLOTS_DIR, exist_ok=True)
print('Module 21 Week 10 — F2/F6 recovery, F3 x3 fix, F8 ultra-tight, F1/F5/F7 momentum')

In [ ]:
submitted_x_w1={1:[0.020584,0.969910],2:[0.814691,0.969505],3:[0.376075,0.370839,0.474761],4:[0.369789,0.452786,0.367951,0.448446],5:[0.241041,0.805036,0.948951,0.905090],6:[0.466959,0.356875,0.489683,0.726384,0.125125],7:[0.027698,0.531762,0.337094,0.176133,0.361503,0.730849],8:[0.192432,0.183093,0.018724,0.036362,0.690267,0.444236,0.081374,0.428967]}
new_y_w1={1:1.966e-321,2:0.1292261555216582,3:-0.010707313301147062,4:-0.34595283782499875,5:1450.9433021815964,6:-0.3611823990070205,7:1.4058168801082682,8:9.8915570907296}
submitted_x_w2={1:[0.591837,0.591837],2:[0.000000,1.000000],3:[0.421053,1.000000,1.000000],4:[0.909548,0.568955,0.762175,0.811807],5:[0.204881,0.877830,0.879582,0.870578],6:[0.851439,0.906254,0.506372,0.594105,0.708147],7:[0.097054,0.432660,0.338116,0.122619,0.296117,0.886436],8:[0.076274,0.101214,0.383035,0.338493,0.113685,0.882235,0.615428,0.796463]}
new_y_w2={1:0.00028209052469858225,2:0.1709619176069506,3:-0.48304244384724265,4:-26.59459580774249,5:1192.2995655092311,6:-1.9259411859252866,7:1.2030170341293975,8:9.0382459830856}
submitted_x_w3={1:[0.980000,0.980000],2:[1.000000,0.306122],3:[1.000000,0.000000,0.684211],4:[0.985601,0.686679,0.243615,0.798556],5:[0.204881,0.877830,0.879582,0.870578],6:[0.061416,0.762464,0.106527,0.271402,0.782742],7:[0.067189,0.412831,0.295130,0.070570,0.412599,0.616173],8:[0.682757,0.427203,0.591529,0.734064,0.514947,0.813984,0.722156,0.615073]}
new_y_w3={1:2.665897212344236e-174,2:-0.042550557700427774,3:-0.1840890683677661,4:-26.07041694623693,5:1192.2995655092311,6:-2.508952125110497,7:1.2533263563752521,8:7.5792591902086}
submitted_x_w4={1:[0.278296,0.020000],2:[0.685269,0.947006],3:[0.403468,0.441923,0.497061],4:[0.352971,0.651614,0.805417,0.616108],5:[0.167299,0.881015,0.978872,0.954244],6:[0.334649,0.293944,0.500782,0.769829,0.074923],7:[0.206363,0.281987,0.389442,0.281544,0.218827,0.711599],8:[0.095545,0.327238,0.051339,0.269531,0.555763,0.417489,0.285113,0.613881]}
new_y_w4={1:-2.6647756688938686e-133,2:0.14705786268424045,3:-0.022992940111015336,4:-0.1283640964538999,5:2496.347187728138,6:-0.38592078647528016,7:2.6705394912160187,8:9.8967959631939}
submitted_x_w5={1:[0.580092,0.683225],2:[0.702813,0.926626],3:[0.365086,0.316421,0.471038],4:[0.344700,0.645505,0.791987,0.622463],5:[0.139557,0.911522,0.979905,0.977049],6:[0.417831,0.356959,0.468069,0.668531,0.039515],7:[0.384097,0.122113,0.444891,0.357064,0.147383,0.783086],8:[0.089787,0.068251,0.180968,0.327284,0.766207,0.653365,0.174832,0.499246]}
new_y_w5={1:0.00008112997850454906,2:0.5833602539566602,3:-0.018707796769607724,4:-13.979947691578896,5:2941.854350298978,6:-0.29985775692426564,7:1.660798293687705,8:9.9275118625839}
submitted_x_w6={1:[0.653384,0.652924],2:[0.704856,0.921380],3:[0.151659,0.826046,0.622768],4:[0.291709,0.714786,0.911879,0.664486],5:[0.102387,0.951309,0.977662,0.978193],6:[0.394360,0.399099,0.411100,0.595076,0.020599],7:[0.027785,0.208441,0.270801,0.266138,0.195016,0.698660],8:[0.029320,0.285338,0.223021,0.041430,0.625367,0.719031,0.033888,0.797446]}
new_y_w6={1:0.0880341468685302,2:0.6478060146282238,3:-0.0867832750698683,4:-21.252967893168208,5:3303.918327634855,6:-0.5181323257010382,7:2.1498244177996053,8:9.8116383300479}
submitted_x_w7={1:[0.533625,0.533688],2:[0.702665,0.919352],3:[0.267218,0.472728,0.513126],4:[0.341374,0.485949,0.606273,0.478470],5:[0.071951,0.977037,0.979767,0.979593],6:[0.441360,0.279908,0.514886,0.684556,0.020780],7:[0.255243,0.272333,0.253655,0.238536,0.237243,0.658362],8:[0.306263,0.307104,0.109816,0.361473,0.669399,0.417361,0.175248,0.274400]}
new_y_w7={1:3.8800114757386434e-13,2:0.6475701405048025,3:-0.038391302132802174,4:-4.940966055787715,5:3626.8315773030813,6:-0.33902767001927475,7:2.5915976846312665,8:9.8171020976195}
submitted_x_w8={1:[0.643797,0.704343],2:[0.703797,0.924514],3:[0.100454,0.240875,0.185908],4:[0.383806,0.511512,0.656499,0.491038],5:[0.044074,0.979912,0.978237,0.978721],6:[0.409346,0.360704,0.502905,0.718263,0.020264],7:[0.138618,0.329866,0.325614,0.264255,0.295279,0.651123],8:[0.182189,0.037358,0.268170,0.170921,0.585250,0.468337,0.247759,0.632930]}
new_y_w8={1:-0.00011412865302137135,2:0.6409162660536529,3:-0.14148000083888568,4:-7.1627894437582,5:3632.183153202294,6:-0.2037089488415273,7:2.7440435657471656,8:9.887359571982}
submitted_x_w9={1:[0.651500,0.654000],2:[0.706000,0.921000],3:[0.020793,0.975360,0.381751],4:[0.354000,0.650000,0.806000,0.617000],5:[0.027000,0.980000,0.979500,0.979000],6:[0.420721,0.410509,0.539518,0.760163,0.022635],7:[0.194674,0.230232,0.307303,0.258261,0.265579,0.680946],8:[0.090500,0.066000,0.182000,0.330000,0.768000,0.650000,0.175000,0.502000]}
new_y_w9={1:0.09715493565789202,2:0.5975456023703872,3:-0.05325899029920575,4:-14.466436970145782,5:3651.372623492115,6:-0.28461851247025494,7:2.9422351452635813,8:9.9270291}
all_time_best={1:(0.09715493565789202,'W9'),2:(0.6478060146282238,'W6'),3:(-0.010707313301147062,'W1'),4:(-0.1283640964538999,'W4'),5:(3651.372623492115,'W9'),6:(-0.2037089488415273,'W8'),7:(2.9422351452635813,'W9'),8:(9.9275118625839,'W5')}
print('Historical data loaded W1-W9')
for i in range(1,9):
    v,w=all_time_best[i]; print(f'  F{i}: {v:.4e} ({w})')

In [ ]:
base_path='/Users/luckydhanvi/Documents/DataScience/PracticalLive/LEVEL_2/imperial-aiml-capstone/data/'
descriptions={1:'Radiation Detection',2:'Noisy ML Model',3:'Drug Discovery',4:'Warehouse Placement',5:'Chemical Yield (STAR)',6:'Cake Recipe',7:'ML Hyperparameters',8:'Complex 8D'}
data={}
for i in range(1,9):
    X0=np.load(f'{base_path}function_{i}/initial_inputs.npy')
    Y0=np.load(f'{base_path}function_{i}/initial_outputs.npy')
    X_all=np.vstack([X0,np.array(submitted_x_w1[i]).reshape(1,-1),np.array(submitted_x_w2[i]).reshape(1,-1),np.array(submitted_x_w3[i]).reshape(1,-1),np.array(submitted_x_w4[i]).reshape(1,-1),np.array(submitted_x_w5[i]).reshape(1,-1),np.array(submitted_x_w6[i]).reshape(1,-1),np.array(submitted_x_w7[i]).reshape(1,-1),np.array(submitted_x_w8[i]).reshape(1,-1),np.array(submitted_x_w9[i]).reshape(1,-1)])
    Y_all=np.concatenate([Y0,[new_y_w1[i]],[new_y_w2[i]],[new_y_w3[i]],[new_y_w4[i]],[new_y_w5[i]],[new_y_w6[i]],[new_y_w7[i]],[new_y_w8[i]],[new_y_w9[i]]])
    data[i]={'X':X_all,'Y':Y_all}
print('W1-W9 data loaded (initial + 9 weekly queries per function)')
for i in range(1,9): print(f'  F{i}: N={len(data[i]["Y"])}')

In [ ]:
BOUND_LO,BOUND_HI=0.02,0.98
all_submitted_X={i:[np.array(submitted_x_w1[i]),np.array(submitted_x_w2[i]),np.array(submitted_x_w3[i]),np.array(submitted_x_w4[i]),np.array(submitted_x_w5[i]),np.array(submitted_x_w6[i]),np.array(submitted_x_w7[i]),np.array(submitted_x_w8[i]),np.array(submitted_x_w9[i])] for i in range(1,9)}

def check_duplicate(next_x,func_num,threshold=0.015):
    return any(np.linalg.norm(next_x-x)<threshold for x in all_submitted_X[func_num])

def expected_improvement(mu,sigma,best_y_t,xi=0.01):
    imp=mu-best_y_t-xi; Z=imp/(sigma+1e-9)
    ei=imp*norm.cdf(Z)+sigma*norm.pdf(Z); ei[sigma<1e-10]=0.0; return ei

def gp_predict_scalar(gp,x):
    return float(gp.predict(x.reshape(1,-1)).ravel()[0])

def build_search_grid(dim,trust_center=None,trust_radius=None,use_lhs=False,n=50000):
    N=n
    if trust_center is not None and trust_radius is not None:
        lo=np.clip(trust_center-trust_radius,BOUND_LO,BOUND_HI)
        hi=np.clip(trust_center+trust_radius,BOUND_LO,BOUND_HI)
        s=LatinHypercube(d=dim,seed=42).random(n=N)
        return lo+s*(hi-lo),f'Trust-LHS r={trust_radius:.3f} {dim}D {N:,}pts'
    elif dim==2:
        g=np.linspace(BOUND_LO,BOUND_HI,224); XX,YY=np.meshgrid(g,g)
        return np.column_stack([XX.ravel(),YY.ravel()]),'224x224 grid'
    elif use_lhs:
        s=LatinHypercube(d=dim,seed=42).random(n=N)
        return BOUND_LO+s*(BOUND_HI-BOUND_LO),f'LHS {N:,}pts {dim}D'
    else:
        np.random.seed(42); return np.random.uniform(BOUND_LO,BOUND_HI,(N,dim)),f'Random {N:,}pts'

def transform_y(Y,method='log'):
    if method=='log': return np.log(np.abs(Y)+1e-300)*np.sign(Y+1e-300)
    elif method=='yeojohnson': Y_t,_=yeojohnson(Y); return Y_t
    return Y

def analyse_w10(func_num,beta_ucb=2.0,use_ei=True,use_lhs=False,trust_center=None,trust_radius=None,xi=0.01,length_scale=0.2,alpha=1e-6,y_transform='log',policy='explore'):
    X,Y=data[func_num]['X'],data[func_num]['Y']
    dim=X.shape[1]; best_idx=np.argmax(Y)
    best_X=X[best_idx] if trust_center is None else trust_center; best_Y=Y[best_idx]
    print(f'\n{"="*65}')
    print(f'F{func_num} {descriptions[func_num]} | N={len(Y)} BestY={best_Y:.4e} | {policy.upper()}')
    print(f'W9={new_y_w9[func_num]:.4e}')
    print('='*65)
    X_grid,grid_info=build_search_grid(dim,trust_center=best_X if trust_radius else None,trust_radius=trust_radius,use_lhs=use_lhs)
    print(f'  [Grid] {grid_info}')
    Y_t=transform_y(Y,method=y_transform)
    kernel=Matern(length_scale=length_scale,nu=2.5)
    gp=GaussianProcessRegressor(kernel=kernel,alpha=alpha,n_restarts_optimizer=3,normalize_y=True)
    gp.fit(X,Y_t); print(f'  [GP] alpha={alpha}')
    mu,sigma=gp.predict(X_grid,return_std=True)
    best_y_t=float(transform_y(np.array([best_Y]),method=y_transform)[0])
    ucb=mu+beta_ucb*sigma; x_ucb=X_grid[np.argmax(ucb)]
    ei=expected_improvement(mu,sigma,best_y_t,xi=xi); x_ei=X_grid[np.argmax(ei)]
    mu_u=gp_predict_scalar(gp,x_ucb); mu_e=gp_predict_scalar(gp,x_ei)
    next_x,winner=(x_ei,'EI') if (use_ei and mu_e>=mu_u) else (x_ucb,'UCB')
    print(f'  [Ensemble] => {winner}')
    if check_duplicate(next_x,func_num):
        np.random.seed(99); next_x=np.clip(next_x+np.random.uniform(-0.02,0.02,dim),BOUND_LO,BOUND_HI); print('  [Dup] Perturbed')
    print(f'  [Dist from best] {np.linalg.norm(next_x-X[best_idx]):.4f}')
    portal='-'.join([f'{v:.6f}' for v in next_x])
    print(f'\n  >>> SUBMIT F{func_num}: {portal} <<<')
    return next_x,portal

print('Helpers ready')

## F1 — Radiation Detection (2D)
**Strategy: MOMENTUM** — W9 new best 0.09715 at (0.6515, 0.6540). Tight r=0.02 to exploit this region.

In [ ]:
W9_BEST_X1=np.array([0.651500,0.654000])
next_x1,portal1=analyse_w10(1,beta_ucb=0.3,use_ei=True,trust_center=W9_BEST_X1,trust_radius=0.02,xi=0.001,policy='momentum')
# Manual override: GP drifted x2 to 0.674 — pull back to confirmed W9 neighbourhood (x2=0.654)
# Sterling/Mark peer insight: F1 may have a spike the kernel averages away; micro-exploit exact best
portal1='0.651000-0.654500'
next_x1=np.array([0.651000,0.654500])
print(f'F1 MANUAL OVERRIDE: {portal1} (GP x2 drifted to 0.674 — corrected near W9 best x2=0.654)')

## F2 — Noisy ML Model (2D)
**Strategy: RECOVERY** — W9 moved x1 to 0.706 (from W6 best 0.7049) and dropped from 0.6478 → 0.5975. Function is very noisy. Return tightly to W6 best coordinates.

In [ ]:
W6_BEST_X2=np.array([0.704856,0.921380])
next_x2,portal2=analyse_w10(2,beta_ucb=0.2,use_ei=True,trust_center=W6_BEST_X2,trust_radius=0.015,xi=0.005,policy='recovery')

## F3 — Drug Discovery (3D)
**Strategy: ANCHOR-TIGHT** — W9 at (0.021, 0.975, 0.382) gave -0.0533. W1 best (0.021, 0.970, 0.475) gave -0.0107. x3 was 0.382 vs W1's 0.475 — fix x3. Tighter r=0.06.

In [ ]:
W1_BEST_X3=np.array([0.020584,0.969910,0.474761])
next_x3,portal3=analyse_w10(3,beta_ucb=0.8,use_ei=True,use_lhs=True,trust_center=W1_BEST_X3,trust_radius=0.06,xi=0.001,policy='anchor-tight')

## F4 — Warehouse Placement (4D)
**Strategy: RECENT-ONLY DYNAMIC** — F4 may be non-stationary (dynamic landscape hypothesis from peer review). Near-exact W4 coords gave -14.466 in W9. Switch to recent-only GP: initial data + W4, W7, W8, W9 only. Skip W2/W3 outliers AND W5/W6 regressions (possibly stale landscape). Trust r=0.05, Yeo-Johnson.

In [ ]:
W4_BEST_X4=np.array([0.352971,0.651614,0.805417,0.616108])

# Recent-only GP: initial data + W4, W7, W8, W9
# Skip W2/W3 (outliers: -26.59, -26.07) AND W5/W6 (regressions: -13.98, -21.25 — possibly stale landscape)
init_X4=np.load('/Users/luckydhanvi/Documents/DataScience/PracticalLive/LEVEL_2/imperial-aiml-capstone/data/function_4/initial_inputs.npy')
init_Y4=np.load('/Users/luckydhanvi/Documents/DataScience/PracticalLive/LEVEL_2/imperial-aiml-capstone/data/function_4/initial_outputs.npy')
mask4=init_Y4>-20  # filter extreme outliers from initial data
recent_X4=np.array([
    [0.352971,0.651614,0.805417,0.616108],  # W4 → -0.1284 (all-time best)
    [0.341374,0.485949,0.606273,0.478470],  # W7 → -4.94
    [0.383806,0.511512,0.656499,0.491038],  # W8 → -7.163
    [0.354000,0.650000,0.806000,0.617000],  # W9 → -14.466
])
recent_Y4=np.array([-0.1284,-4.94,-7.163,-14.466])
data[4]['X']=np.vstack([init_X4[mask4],recent_X4])
data[4]['Y']=np.concatenate([init_Y4[mask4],recent_Y4])
print(f'F4 recent-only dataset: N={len(data[4]["Y"])} (initial clean + W4/W7/W8/W9)')

next_x4,portal4=analyse_w10(4,beta_ucb=0.5,use_ei=True,use_lhs=True,trust_center=W4_BEST_X4,trust_radius=0.05,xi=0.01,alpha=0.1,y_transform='yeojohnson',policy='recent-only-dynamic')

## F5 — Chemical Yield STAR (4D)
**Strategy: MOMENTUM** — W9 new best 3651.37 at x1=0.027. x1 trend: 0.241→0.140→0.102→0.072→0.044→0.027→**~0.015**. x2-x4 stay at 0.979-0.980.

In [ ]:
W9_BEST_X5=np.array([0.027000,0.980000,0.979500,0.979000])
next_x5,portal5=analyse_w10(5,beta_ucb=0.02,use_ei=True,use_lhs=True,trust_center=W9_BEST_X5,trust_radius=0.015,xi=0.01,policy='momentum')
# Manual override: push x1 ridge further down (GP can perturb backwards due to duplicate check)
# Ridge trend: 0.241→0.140→0.102→0.072→0.044→0.027→0.015→0.010
portal5='0.010000-0.980000-0.979500-0.979000'
next_x5=np.array([0.010000,0.980000,0.979500,0.979000])
print(f'F5 MANUAL OVERRIDE: {portal5} (ridge x1: 0.027→0.015→0.010, x2-x4 held at 0.979-0.980)')

## F6 — Cake Recipe (5D)
**Strategy: RECOVERY** — W9 pushed x4 to 0.760 and regressed to -0.2846 (from W8 best -0.2037). Return tight to W8 best. x4 increasing was the wrong direction.

In [ ]:
W8_BEST_X6=np.array([0.409346,0.360704,0.502905,0.718263,0.020264])
next_x6,portal6=analyse_w10(6,beta_ucb=0.2,use_ei=True,use_lhs=True,trust_center=W8_BEST_X6,trust_radius=0.03,xi=0.001,policy='recovery')

## F7 — ML Hyperparameters (6D)
**Strategy: MOMENTUM** — W9 new best 2.9422. Strong trend: 2.150→2.592→2.744→2.942. Continue from W9 best, tighter r=0.08.

In [ ]:
W9_BEST_X7=np.array([0.194674,0.230232,0.307303,0.258261,0.265579,0.680946])
next_x7,portal7=analyse_w10(7,beta_ucb=0.5,use_ei=True,use_lhs=True,trust_center=W9_BEST_X7,trust_radius=0.08,xi=0.01,policy='momentum')

## F8 — Complex 8D
**Strategy: ULTRA-TIGHT RECOVERY** — W9 at 9.9270 missed W5 best (9.9275) by 0.0005. W9 had x2=0.066 vs W5's 0.068251. Tightest radius yet: r=0.04 around W5 exact best.

In [ ]:
W5_BEST_X8=np.array([0.089787,0.068251,0.180968,0.327284,0.766207,0.653365,0.174832,0.499246])
next_x8,portal8=analyse_w10(8,beta_ucb=0.3,use_ei=True,use_lhs=True,trust_center=W5_BEST_X8,trust_radius=0.04,xi=0.01,policy='ultra-tight-recovery')

In [ ]:
print('='*70)
print('WEEK 10 — MODULE 21 — FINAL PORTAL SUBMISSION STRINGS')
print('='*70)
# Manual overrides applied:
#   F1: GP x2 drifted to 0.674 — corrected to 0.654500 (near W9 best)
#   F4: Recent-only GP (W4+W7+W8+W9) — dynamic landscape hypothesis
#   F5: x1 ridge pushed 0.015→0.010 (trend: 0.241→0.140→...→0.027→0.015→0.010)
portals={1:portal1,2:portal2,3:portal3,4:portal4,5:portal5,6:portal6,7:portal7,8:portal8}
policies={
    1:'MOMENTUM (manual — x2 corrected)',
    2:'RECOVERY (W6 best, r=0.015)',
    3:'ANCHOR-TIGHT (W1 best, x3 corrected)',
    4:'RECENT-ONLY DYNAMIC (W4+W7+W8+W9, r=0.05)',
    5:'MOMENTUM (manual — x1 ridge 0.027→0.010)',
    6:'RECOVERY (W8 best, r=0.03)',
    7:'MOMENTUM (W9 best, r=0.08)',
    8:'ULTRA-TIGHT (W5 best, r=0.04)',
}
for i in range(1,9): print(f'F{i}: {portals[i]}  [{policies[i]}]')
print()
print('All-time bests (post-W9):')
for i in range(1,9):
    v,w=all_time_best[i]; print(f'  F{i}: best={v:.4e} ({w})')

In [ ]:
weekly_results_all={
    1:[new_y_w1[i] for i in range(1,9)],2:[new_y_w2[i] for i in range(1,9)],
    3:[new_y_w3[i] for i in range(1,9)],4:[new_y_w4[i] for i in range(1,9)],
    5:[new_y_w5[i] for i in range(1,9)],6:[new_y_w6[i] for i in range(1,9)],
    7:[new_y_w7[i] for i in range(1,9)],8:[new_y_w8[i] for i in range(1,9)],
    9:[new_y_w9[i] for i in range(1,9)]
}
fig,axes=plt.subplots(2,4,figsize=(18,8))
fig.suptitle('Capstone W1-W9 Progress (Module 21)',fontsize=14,fontweight='bold')
for idx,fn in enumerate(range(1,9)):
    ax=axes[idx//4][idx%4]; weeks=list(range(1,10))
    vals=[weekly_results_all[w][fn-1] for w in weeks]
    running_best=[max(vals[:w]) for w in range(1,len(vals)+1)]
    ax.plot(weeks,vals,'o--',color='steelblue',alpha=0.7,label='Query')
    ax.plot(weeks,running_best,'s-',color='darkorange',linewidth=2,label='Best')
    best_v,best_w=all_time_best[fn]
    ax.set_title(f'F{fn}: {descriptions[fn]}\nbest={best_v:.3e} ({best_w})',fontsize=8)
    ax.set_xlabel('Week'); ax.set_ylabel('Output'); ax.legend(fontsize=7); ax.grid(True,alpha=0.3)
plt.tight_layout()
plot_path=os.path.join(PLOTS_DIR,'w9_progress_analysis.png')
plt.savefig(plot_path,dpi=150,bbox_inches='tight'); plt.close()
print(f'Plot saved: {plot_path}')